# Amenity Index Creation

Here I'll be combining data from the `city_facilities_data_clean_analysis.ipynb` and `sf_acs_walkscore.ipynb` notebooks to construct an index:

In [4]:
# import necessary dependencies 
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
from census import Census
from us import states
import os
from pathlib import Path
from IPython.display import display
import seaborn as sns
import re
import zipfile
from shapely.geometry import Point
import folium
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib as mpl

# set general seaborn style
sns.set_theme(style="whitegrid", font_scale=1.1)

In [10]:
# uploading processed data from previous notebooks

# use robust path handling to locate the data file
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'requirements.txt').exists() or (p / '.git').exists():
            return p
    return start

repo_root = find_repo_root()

amenities_gdf = gpd.read_file(repo_root / 'data' / 'processed' / 'city_facilities_cleaned.geojson', engine="pyogrio")
walkscore_census_df = gpd.read_file(repo_root / 'data' / 'processed' / 'final_sf_acs_years_panel.csv', engine="pyogrio")

#### Combine Data 

First, I'll take another glance at the data to be able to combine them by tract, so I can actually create an index.

I'll start with looking at the `amenities_gdf` file.

In [ ]:
# see the first few rows of amenities_gdf
amenities_gdf.head()

,:id,:version,facility_id,common_name,address,city,zip_code,block_lot,owned_leased,dept_id,...,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,amenity_category,geometry
0,row-s7n8_dawh~di6g,rv-cqxy.cfcc_9h38,3199,SE CENTRIFUGAL BLDG - 840,1800 Oakdale Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39422 37.73832)
1,row-bhtc-9uc2-x9yi,rv-65e3-cxdx.w4qh,1348,Wholesale Produce Market - Public Dock 2 Middle,2002 Jerrold Ave,San Francisco,94124,5281020,Own,63,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39736 37.74330)
2,row-y6zf-u23p.6mxm,rv-4rqu.rsa9~zrey,3181,SE SED BLDG #3 042,1700 Jerrold Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.39112 37.74330)
3,row-jx69-xpuh-gzwy,rv-ujtr-73h5-tc8h,3187,SE D. SLUDGE THK. TANK - 750,1700 Jerrold Ave,San Francisco,94124,5313002,Own,47,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.38981 37.74273)
4,row-gsda_jc6m~mcgx,rv-6d7q~5xph-y6a9,1808,Urban Forestry Crew Shack,2323 Cesar Chavez St,San Francisco,94124,4341001,Own,77,...,9809,Census Tract 9809,G5020,S,3602295,258160,+37.7462860,-122.3894769,Infrastructure,POINT (-122.40040 37.74905)


In [ ]:
# see the columns of amenities_gdf
amenities_gdf.columns

Index([':id', ':version', 'facility_id', 'common_name', 'address', 'city',
       'zip_code', 'block_lot', 'owned_leased', 'dept_id', 'department_name',
       'gross_sq_ft', 'longitude', 'latitude', 'supervisor_district',
       'city_tenants', 'land_id', ':@computed_region_ajp5_b2md',
       ':@computed_region_f58d_8dbm', ':@computed_region_rxqg_mtj9',
       ':@computed_region_jx4q_fizf', ':@computed_region_yftq_j783',
       ':@computed_region_bh8s_q3mv', ':@computed_region_jwn9_ihcz',
       ':@computed_region_6qbp_sg9q', ':@computed_region_qgnn_b9vv',
       ':@computed_region_26cr_cadq', 'index_right', 'STATEFP', 'COUNTYFP',
       'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT',
       'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'amenity_category',
       'geometry'],
      dtype='object')

In [12]:
# see the shape of amenities_gdf
amenities_gdf.shape

(1316, 43)

In [29]:
# see the number of unique tracts in amenities_gdf
amenities_gdf['GEOID'].nunique()

209

Now I'll move onto taking a look at the `walkscore_census_df` file.

In [14]:
# see the first few rows of walkscore_census_df
walkscore_census_df.head()

,NAME,population_total,white/non_hispanic,gini_index,median_income,poverty_denominator,poverty_count,transit_commuters,state,county,tract,GEOID,year,%_white_nonhisp,poverty_rate
0,"Census Tract 260.04, San Francisco County, Cal...",5015.0,528.0,0.4582,65862.0,5015.0,456.0,1099.0,06,075,026004,06075026004,2015,0.10528414755732801,0.0909272183449651
1,"Census Tract 301.01, San Francisco County, Cal...",4895.0,2915.0,0.4959,79071.0,4735.0,583.0,1326.0,06,075,030101,06075030101,2015,0.5955056179775281,0.12312565997888067
2,"Census Tract 330, San Francisco County, Califo...",8227.0,2891.0,0.4607,82527.0,8162.0,1088.0,1093.0,06,075,033000,06075033000,2015,0.3514039139418986,0.1333006616025484
3,"Census Tract 254.03, San Francisco County, Cal...",5154.0,1168.0,0.4495,73159.0,5128.0,611.0,1018.0,06,075,025403,06075025403,2015,0.22662010089251067,0.11914976599063963
4,"Census Tract 264.01, San Francisco County, Cal...",3937.0,137.0,0.4317,46150.0,3937.0,501.0,624.0,06,075,026401,06075026401,2015,0.034798069596139194,0.12725425450850902


In [15]:
# see columns of walkscore_census_df
walkscore_census_df.columns

Index(['NAME', 'population_total', 'white/non_hispanic', 'gini_index',
       'median_income', 'poverty_denominator', 'poverty_count',
       'transit_commuters', 'state', 'county', 'tract', 'GEOID', 'year',
       '%_white_nonhisp', 'poverty_rate'],
      dtype='object')

In [17]:
# see the shape of walkscore_census_df
walkscore_census_df.shape

(1088, 15)

In [20]:
# see the years available in walkscore_census_df
walkscore_census_df['year'].value_counts()

year
2020    236
2022    236
2023    236
2015    190
2018    190
Name: count, dtype: int64

We'll use the data from the most recent year (2020) to merge with amenity related features.

In [32]:
# get walkscore census data from 2020
walkscore_2020_df = walkscore_census_df[walkscore_census_df['year'] == "2020"]

In [ ]:
amenities_gdf[["GEOID", "gross_sq_ft", "amenity_category"]].merge(walkscore_2020_df, on="GEOID")

KeyError: ('GEOID', 'gross_sq_ft', 'amenity_category')